# GAP-ITSM v2: Fix A+C — Drop SL=0.75 + Fixed Contracts
## CSCV/PBO Overfitting Analysis

RCA-driven improvements to address PBO=58.2%:
- **Fix A**: Remove SL=0.75 from CSCV grid (consistently worst, wins IS only 1.3%)
- **Fix C**: Fixed 2 contracts (removes leverage differential between SL values)
  - Root cause: risk-scaled sizing made tight-SL amplify IS-period trends -> not OOS-stable
  - Fixed contracts = pure SL-distance comparison, regime-invariant leverage
- CSCV grid: sl=[0.5,1.0] x vol=[42,63,126] = N=6 variants

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
sys.path.insert(0, os.path.abspath('.'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
INITIAL_EQUITY = 50_000.0
FIXED_CONTRACTS = 2

In [2]:
csv = os.path.join(ROOT, 'data', 'NQ_5m.csv')
df = pd.read_csv(csv, index_col=0)
df.index = pd.to_datetime(df.index, utc=True).tz_convert('America/New_York')
df.columns = [c.lower() for c in df.columns]
print(f'Bars loaded : {len(df):,}')
print(f'Date range  : {df.index[0].date()} to {df.index[-1].date()}')

Bars loaded : 222,295
Date range  : 2014-12-19 to 2026-03-17


In [3]:
from strategy_gap_itsm import run_backtest as gap_bt

def perf(name, result):
    trades, eq = result
    if trades.empty: return None
    dp = trades.groupby('date')['pnl'].sum()
    sr = dp.mean() / dp.std() * np.sqrt(252)
    peak = eq.cummax(); dd = ((eq - peak) / peak).min()
    wr = (trades['pnl'] > 0).mean()
    tr = (trades['equity_after'].iloc[-1] - INITIAL_EQUITY) / INITIAL_EQUITY * 100
    return {'name': name, 'trades': len(trades), 'wr': wr, 'sharpe': sr,
            'dd': dd, 'tr': tr, 'eq': eq}

# Compare: original GAP-ITSM (risk-scaled) vs v2 (fixed contracts)
configs = [
    ('V1: GAP-ITSM (risk-scaled, sl=0.5)',  gap_bt(df, sl_atr_mult=0.5)),
    ('V2: GAP-ITSM (risk-scaled, sl=1.0)',  gap_bt(df, sl_atr_mult=1.0)),
    ('V3: GAP-ITSM v2 (fixed=2, sl=0.5)',   gap_bt(df, sl_atr_mult=0.5, fixed_contracts=FIXED_CONTRACTS)),
    ('V4: GAP-ITSM v2 (fixed=2, sl=1.0)',   gap_bt(df, sl_atr_mult=1.0, fixed_contracts=FIXED_CONTRACTS)),
]
results = [perf(n, r) for n, r in configs]

hdr = f'{"Strategy":<40} {"Trades":>7} {"WinRate":>8} {"Sharpe":>8} {"MaxDD":>8} {"TotalRet":>10}'
print(hdr)
for r in results:
    if r: print(f'{r["name"]:<40} {r["trades"]:>7} {r["wr"]:>8.1%} {r["sharpe"]:>8.3f} {r["dd"]:>8.1%} {r["tr"]:>9.1f}%')
print(f'  S&P 500 benchmark                           -        -    0.600  -55.0%     170.0%')

Strategy                                  Trades  WinRate   Sharpe    MaxDD   TotalRet
V1: GAP-ITSM (risk-scaled, sl=0.5)           457    54.0%    1.987   -13.3%      49.0%
V2: GAP-ITSM (risk-scaled, sl=1.0)           457    58.0%    1.684    -9.0%      28.5%
V3: GAP-ITSM v2 (fixed=2, sl=0.5)            457    54.0%    2.006    -8.0%      58.6%
V4: GAP-ITSM v2 (fixed=2, sl=1.0)            457    58.0%    2.021    -9.4%      60.1%
  S&P 500 benchmark                           -        -    0.600  -55.0%     170.0%


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2196F3','#FF9800','#4CAF50','#F44336']
ax = axes[0]
for i, r in enumerate(results):
    if r and r['eq'] is not None:
        eq = r['eq']
        ax.plot(eq.index, (eq / INITIAL_EQUITY - 1) * 100, label=r['name'][:25], color=colors[i], linewidth=1.5)
try:
    import yfinance as yf
    spy = yf.download('SPY', start='2015-01-01', end='2026-03-17', auto_adjust=True, progress=False)
    spy_r = (spy['Close'] / spy['Close'].iloc[0] - 1) * 100
    ax.plot(spy_r.index, spy_r, 'k--', label='S&P 500', linewidth=1.0, alpha=0.6)
except: pass
ax.set_title('Risk-Scaled vs Fixed Contracts'); ax.set_ylabel('Return (%)')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
ax2 = axes[1]
sharpes = [r['sharpe'] for r in results if r]
dds = [-r['dd']*100 for r in results if r]
x = np.arange(len(sharpes))
ax2.bar(x - 0.2, sharpes, 0.35, label='Sharpe', color=colors[:len(sharpes)])
ax2.bar(x + 0.2, dds, 0.35, label='Max DD %', color=colors[:len(sharpes)], alpha=0.5, hatch='/')
ax2.axhline(1.0, color='green', linestyle='--', alpha=0.8, label='Sharpe=1.0')
ax2.axhline(10.0, color='red', linestyle='--', alpha=0.8, label='DD=10%')
ax2.set_xticks(x)
ax2.set_xticklabels(['RS sl=0.5','RS sl=1.0','FC sl=0.5','FC sl=1.0'], fontsize=8)
ax2.set_title('Sharpe vs Max DD (RS=risk-scaled, FC=fixed-contracts)')
ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'output', 'gap_itsm_v2_backtest.png'), dpi=120, bbox_inches='tight')
plt.show(); print('Saved gap_itsm_v2_backtest.png')

Saved gap_itsm_v2_backtest.png


In [5]:
from cscv_gap_itsm import build_pnl_matrix, run_cscv, pbo_verdict

# Fix A: drop SL=0.75 | Fix C: fixed 2 contracts
SL_MULTS = [0.5, 1.0]
VOL_LBS  = [42, 63, 126]
variants = [{'sl_atr_mult': sl, 'vol_lookback': vl, 'fixed_contracts': FIXED_CONTRACTS}
            for sl in SL_MULTS for vl in VOL_LBS]
N = len(variants)
print(f'N = {N} variants (Fix A: no SL=0.75, Fix C: fixed_contracts={FIXED_CONTRACTS})')
for v in variants:
    print(f'  SL={v["sl_atr_mult"]}xATR  vol_lookback={v["vol_lookback"]}d  contracts={v["fixed_contracts"]}')

N = 6 variants (Fix A: no SL=0.75, Fix C: fixed_contracts=2)
  SL=0.5xATR  vol_lookback=42d  contracts=2
  SL=0.5xATR  vol_lookback=63d  contracts=2
  SL=0.5xATR  vol_lookback=126d  contracts=2
  SL=1.0xATR  vol_lookback=42d  contracts=2
  SL=1.0xATR  vol_lookback=63d  contracts=2
  SL=1.0xATR  vol_lookback=126d  contracts=2


In [6]:
print(f'Building PnL matrix ({N} variants)...')
M, dates = build_pnl_matrix(df, variants)
print(f'Shape: {M.shape}  (T={M.shape[0]} days, N={M.shape[1]} variants)')
print(f'Range: {dates[0].date()} to {dates[-1].date()}')

Building PnL matrix (6 variants)...


  3/6 variants complete...


  6/6 variants complete...
Shape: (551, 6)  (T=551 days, N=6 variants)
Range: 2014-12-24 to 2026-03-16


In [7]:
print('Running CSCV  S=16  C(16,8)=12,870 combinations...')
res = run_cscv(M, S=16)
pbo = res['pbo']
print()
print(f'PBO  = {pbo:.1%}  {pbo_verdict(pbo)}')
print(f'Median IS Sharpe  : {np.median(res["is_sharpes"]):.3f}')
print(f'Median OOS Sharpe : {np.median(res["oos_sharpes"]):.3f}')
deg = (np.median(res['is_sharpes']) - np.median(res['oos_sharpes'])) / max(np.median(res['is_sharpes']), 1e-6) * 100
print(f'Sharpe degradation: {deg:.1f}%')
print(f'Prob(loss OOS)    : {np.mean(res["oos_sharpes"] < 0):.1%}')
print()
print('Comparison:')
print(f'  ORB v4 (60 variants, risk-scaled)     : 47.7%  MODERATE')
print(f'  ITSM-ATR (9 variants, risk-scaled)    : 60.6%  HIGH')
print(f'  GAP-ITSM v1 (9 variants, risk-scaled) : 58.2%  HIGH')
print(f'  GAP-ITSM v2 (6 variants, fix A+C)     : {pbo:.1%}  {pbo_verdict(pbo)}')

Running CSCV  S=16  C(16,8)=12,870 combinations...



PBO  = 59.5%  HIGH OVERFITTING RISK (PBO >= 50%)
Median IS Sharpe  : 2.211
Median OOS Sharpe : 1.652
Sharpe degradation: 25.3%
Prob(loss OOS)    : 1.4%

Comparison:
  ORB v4 (60 variants, risk-scaled)     : 47.7%  MODERATE
  ITSM-ATR (9 variants, risk-scaled)    : 60.6%  HIGH
  GAP-ITSM v1 (9 variants, risk-scaled) : 58.2%  HIGH
  GAP-ITSM v2 (6 variants, fix A+C)     : 59.5%  HIGH OVERFITTING RISK (PBO >= 50%)


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
logits = res['logits']
ax.hist(logits, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='red', linewidth=1.5, linestyle='--', label='PBO boundary')
ax.set_title(f'Logit Distribution  PBO={np.mean(logits<0)*100:.1f}%')
ax.set_xlabel('Logit'); ax.set_ylabel('Frequency')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
is_sr = res['is_sharpes']; oos_sr = res['oos_sharpes']
ax.scatter(is_sr, oos_sr, alpha=0.1, s=4, color='steelblue')
vmin = min(is_sr.min(), oos_sr.min()); vmax = max(is_sr.max(), oos_sr.max())
ax.plot([vmin, vmax], [vmin, vmax], 'k--', linewidth=1, label='IS=OOS (ideal)')
ax.axhline(0, color='red', linewidth=0.8, linestyle=':')
ax.set_xlabel('IS Sharpe'); ax.set_ylabel('OOS Sharpe')
ax.set_title('IS vs OOS Sharpe (v2)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Per-variant PBO contribution
ax = axes[2]
var_labels = [f'SL={v["sl_atr_mult"]}\nvl={v["vol_lookback"]}' for v in variants]
pbo_by_var = []
for i, v in enumerate(variants):
    sel = res['selected_variants']
    logit_v = res['logits'][sel == i]
    pbo_v = np.mean(logit_v < 0) if len(logit_v) > 0 else 0.5
    pbo_by_var.append(pbo_v * 100)
ax.bar(range(N), pbo_by_var, color=['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0','#00BCD4'])
ax.axhline(20, color='green', linestyle='--', alpha=0.8, label='20% target')
ax.axhline(50, color='red', linestyle='--', alpha=0.6)
ax.set_xticks(range(N)); ax.set_xticklabels(var_labels, fontsize=7)
ax.set_title('PBO When Each Variant is IS Winner')
ax.set_ylabel('PBO (%)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'output', 'gap_itsm_v2_cscv.png'), dpi=120, bbox_inches='tight')
plt.show(); print('Saved gap_itsm_v2_cscv.png')

Saved gap_itsm_v2_cscv.png


In [9]:
# Year-by-year winner analysis: does fixed contracts stabilize ordering?
years = sorted(set(df.index.year))
yearly_sharpes = {f'SL={v["sl_atr_mult"]} vl={v["vol_lookback"]}': {} for v in variants}

for v in variants:
    t, _ = gap_bt(df, **v)
    if t.empty: continue
    t['year'] = pd.to_datetime(t['date']).dt.year
    for yr in years:
        yt = t[t['year'] == yr]
        if len(yt) < 5: continue
        dp = yt.groupby('date')['pnl'].sum()
        sr = dp.mean() / dp.std() * np.sqrt(252) if dp.std() > 0 else 0
        vname = f'SL={v["sl_atr_mult"]} vl={v["vol_lookback"]}'
        yearly_sharpes[vname][yr] = round(sr, 3)

print(f'{"Year":<6}', end='')
for v in variants:
    print(f'  SL={v["sl_atr_mult"]} vl={v["vol_lookback"]}', end='')
print()
stable_years = 0
for yr in years:
    row = []
    for v in variants:
        vname = f'SL={v["sl_atr_mult"]} vl={v["vol_lookback"]}'
        row.append(yearly_sharpes[vname].get(yr, float('nan')))
    valid = [(r, i) for i, r in enumerate(row) if not np.isnan(r)]
    if not valid: continue
    winner_idx = max(valid, key=lambda x: x[0])[1]
    winner_v = variants[winner_idx]
    print(f'{yr:<6}', end='')
    for r in row: print(f'  {r:8.3f}' if not np.isnan(r) else '       -', end='')
    print(f'  -> SL={winner_v["sl_atr_mult"]} vl={winner_v["vol_lookback"]}')

Year    SL=0.5 vl=42  SL=0.5 vl=63  SL=0.5 vl=126  SL=1.0 vl=42  SL=1.0 vl=63  SL=1.0 vl=126
2015      -1.177    -1.896    -0.583    -1.820    -2.477    -1.086  -> SL=0.5 vl=126
2016       1.298     0.591     0.795     0.846     0.034    -0.034  -> SL=0.5 vl=42
2017       3.207     5.827     5.583     1.101     4.317     5.001  -> SL=0.5 vl=63
2018       4.981     6.000     5.343     7.941     9.426     8.958  -> SL=1.0 vl=63
2019       0.142    -1.964     0.352    -0.378    -0.360     0.000  -> SL=0.5 vl=126
2020       0.439     0.720    -0.085     3.154     2.598     1.351  -> SL=1.0 vl=42
2021       6.783     6.732     4.848     7.108     6.134     5.368  -> SL=1.0 vl=42
2022       2.588     2.237     2.343     3.617     3.372     3.284  -> SL=1.0 vl=42
2023       2.083     2.726     2.411     3.897     4.377     3.360  -> SL=1.0 vl=63
2024       4.027     5.032     4.931     2.056     2.793     2.782  -> SL=0.5 vl=63
2025      -0.183     0.162    -1.567    -0.673    -1.015    -1.83

In [10]:
print('Per-variant full-dataset performance (Fix A+C):')
print(f'{"Variant":<30} {"Trades":>7} {"Sharpe":>8} {"MaxDD":>8} {"Sharpe>1":>9} {"DD<10%":>8}')
pass_both = 0
for v in variants:
    t, eq = gap_bt(df, **v)
    if t.empty: continue
    dp = t.groupby('date')['pnl'].sum()
    sr = dp.mean() / dp.std() * np.sqrt(252)
    peak = eq.cummax(); dd = ((eq - peak) / peak).min()
    ps = 'PASS' if sr > 1.0 else 'FAIL'
    pd_ = 'PASS' if dd > -0.10 else 'FAIL'
    if ps == 'PASS' and pd_ == 'PASS': pass_both += 1
    name = f"SL={v['sl_atr_mult']}  vol={v['vol_lookback']}d  fc={v['fixed_contracts']}"
    print(f'{name:<30} {len(t):>7} {sr:>8.3f} {dd:>8.1%} {ps:>9} {pd_:>8}')
print(f'\nVariants passing BOTH targets: {pass_both}/{len(variants)}')
print()
print('=== FINAL VERDICT (GAP-ITSM v2 Fix A+C) ===')
print(f'PBO = {res["pbo"]:.1%}  --  {pbo_verdict(res["pbo"])}')
print(f'Prob(loss OOS)   : {np.mean(res["oos_sharpes"] < 0):.1%}')
print(f'Median OOS Sharpe: {np.median(res["oos_sharpes"]):.3f}')
print(f'Variants passing Sharpe>1.0 AND DD<10%: {pass_both}/{len(variants)}')

Per-variant full-dataset performance (Fix A+C):
Variant                         Trades   Sharpe    MaxDD  Sharpe>1   DD<10%


SL=0.5  vol=42d  fc=2              428    1.852    -9.4%      PASS     PASS


SL=0.5  vol=63d  fc=2              457    2.006    -8.0%      PASS     PASS


SL=0.5  vol=126d  fc=2             460    1.177   -15.6%      PASS     FAIL


SL=1.0  vol=42d  fc=2              428    2.096    -9.5%      PASS     PASS


SL=1.0  vol=63d  fc=2              457    2.021    -9.4%      PASS     PASS


SL=1.0  vol=126d  fc=2             460    1.529   -12.8%      PASS     FAIL

Variants passing BOTH targets: 4/6

=== FINAL VERDICT (GAP-ITSM v2 Fix A+C) ===
PBO = 59.5%  --  HIGH OVERFITTING RISK (PBO >= 50%)
Prob(loss OOS)   : 1.4%
Median OOS Sharpe: 1.652
Variants passing Sharpe>1.0 AND DD<10%: 4/6
